# Section 1

In [1]:
from data_preparer import LogNoteDataPreparer
from feature_builder import UserActivityFeatureBuilder
import pandas as pd
import numpy as np

# Helper pour encoder les colonnes datetime en jours depuis une origine t0
from typing import Tuple

def encode_datetime_to_days(X_train: pd.DataFrame, X_test: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    dt_cols_train = X_train.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns
    dt_cols_test = X_test.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns
    dt_cols = list(set(dt_cols_train).union(set(dt_cols_test)))
    if not dt_cols:
        return X_train, X_test
    # t0 pris sur l'ensemble (train+test) pour cohérence d'échelle
    t0 = pd.to_datetime(pd.concat([X_train[dt_cols], X_test[dt_cols]], axis=0).stack().min())
    for c in dt_cols:
        if c in X_train.columns:
            X_train[c] = (pd.to_datetime(X_train[c]) - t0).dt.total_seconds() / 86400.0
        if c in X_test.columns:
            X_test[c] = (pd.to_datetime(X_test[c]) - t0).dt.total_seconds() / 86400.0
    return X_train, X_test


In [2]:
obj = LogNoteDataPreparer("logs_info_25_pseudo.csv", "notes_info_25_pseudo.csv")

In [3]:
train_x, test_x, train_y, test_y = obj.split_train_test()


Nombre total de logs : 16227
Nombre initial de notes : 99
Nombre d'étudiants après merge : 95
Nombre d'étudiants en train : 76
Nombre d'étudiants en test : 19


# Section 2

In [4]:
# === Features TRAIN ===
df_train_x = pd.read_csv("train_x.csv")
fb_train = UserActivityFeatureBuilder(df_train_x)
df_train_features = fb_train.build_features()  # contient 'pseudo'

# Charger y_train et ALIGNER
train_y_df = pd.read_csv("train_y.csv")  # colonnes: pseudo, note
train_merged = df_train_features.merge(train_y_df, on="pseudo", how="inner")

# Séparer X/y
X_train = train_merged.drop(columns=["pseudo", "note"])  # garde potentiels datetime
y_train = train_merged["note"]

# === Features TEST ===
df_test_x = pd.read_csv("test_x.csv")
fb_test = UserActivityFeatureBuilder(df_test_x)
df_test_features = fb_test.build_features()

test_y_df = pd.read_csv("test_y.csv")  # colonnes: pseudo, note
test_merged = df_test_features.merge(test_y_df, on="pseudo", how="inner")

X_test = test_merged.drop(columns=["pseudo", "note"])  # garde potentiels datetime
y_test = test_merged["note"]

# Encoder les colonnes datetime -> jours (float)
X_train, X_test = encode_datetime_to_days(X_train, X_test)

# S'assurer que tout est numérique + NaN -> 0
X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0)
X_test = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

# Petits diagnostics
print("X_train shape:", X_train.shape, " | y_train:", y_train.shape)
print("X_test shape:", X_test.shape, " | y_test:", y_test.shape)


ValueError: You are trying to merge on str and int64 columns for key 'pseudo'. If you wish to proceed you should use pd.concat

In [5]:
print("Aperçu features train:")
print(X_train.head())


Aperçu features train:


NameError: name 'X_train' is not defined

# Section 3
## Sous section 1 : Linear Regression

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


In [7]:
lr = LinearRegression()


In [8]:
lr.fit(X_train, y_train)

NameError: name 'X_train' is not defined

In [14]:
y_pred_lr = lr.predict(X_test)

NameError: name 'X_test' is not defined

In [13]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("[LinearRegression] MAE :", mae_lr)
print("[LinearRegression] R2  :", r2_lr)

# Importance par coefficients
importances_lr = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lr.coef_,
})
importances_lr["importance"] = importances_lr["coefficient"].abs()
importances_lr = importances_lr.sort_values("importance", ascending=False)
print("Top coefficients (LR):", importances_lr.head(20))


NameError: name 'y_test' is not defined

## Sous section 2 : RandomForestRegressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf = RandomForestRegressor(random_state=42)

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("[RandomForest] MAE :", mae_rf)
print("[RandomForest] R2  :", r2_rf)


In [ ]:
importances_rf = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)
print("Top importances (RF):", importances_rf.head(20))


# Section 4 : Sauvegarde du modèle LR

In [ ]:
import joblib
joblib.dump(lr, "model_lr.pkl")
print("Modèle LinearRegression sauvegardé dans model_lr.pkl")


# Section 5 (ancien code supprimé)
- L'ancienne section utilisait ClasseB et methodeA, remplacés par UserActivityFeatureBuilder.build_features()
- La prédiction illustrative ci-dessous utilise le modèle LR et X_test déjà préparé

In [ ]:
import matplotlib.pyplot as plt

# Utilisons les prédictions LR pour l'exemple de visualisation
resultats = pd.Series(y_pred_lr >= 10).map({True: "Réussite", False: "Échec"})
counts = resultats.value_counts()

plt.figure()
plt.bar(counts.index, counts.values)
plt.xlabel("Résultat")
plt.ylabel("Nombre d'étudiants")
plt.title("Réussite / Échec selon la prédiction (LinearRegression)")
plt.show()


In [15]:
import json, os

outdir = "outputs"             # ou le dossier de ton choix
os.makedirs(outdir, exist_ok=True)

schema = {"feature_names": list(X_train.columns)}
with open(os.path.join(outdir, "feature_schema.json"), "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

# puis sauvegarde le modèle dans le même dossier
joblib.dump(lr, os.path.join(outdir, "model_lr.pkl"))

NameError: name 'X_train' is not defined